In [ ]:
import scanpy as sc

path_to_adata = '/02_combination_10x_pb/hdf5/20250221_all_data_integrated_filtered_umap_scANVI_metadata_ct_corrected.sn.h5ad'
adata = sc.read_h5ad(path_to_adata)

In [ ]:
adata.obs

In [ ]:
adata.obs.cell_type_simplified.value_counts().to_csv('cell_numbers/sn_number_of_cells.tsv', sep = '\t')

In [ ]:
path_to_adata = '/02_combination_10x_pb/hdf5/20250811_all_data_integrated_filtered_umap_scANVI_metadata_ct_corrected.annot_corrected.index_dedup.cg.h5ad'
adata = sc.read_h5ad(path_to_adata)

In [ ]:
adata.obs.consensus_cell_type_corrected.value_counts().to_csv('cell_numbers/cc_number_of_cells.tsv', sep = '\t')

In [ ]:
import decoupler as dc
import pertpy as pt
import scanpy as sc
from natsort import natsorted
import pandas as pd
import numpy as np
import gzip
import datetime
from sklearn.decomposition import PCA
from sklearn import preprocessing


import warnings
import os
warnings.filterwarnings("ignore")


# load adata
path_to_adata = '/02_combination_10x_pb/hdf5/20250221_all_data_integrated_filtered_umap_scANVI_metadata_ct_corrected.sn.h5ad'
adata = sc.read_h5ad(path_to_adata)
adata.obs.set_index(adata.obs.new_index.astype(str),  inplace=True)

# prepare data for pseudobulk
adata.X = adata.layers["counts"]
adata = adata[~adata.obs['experiment'].isin(['mo005', 'mo009'])]
adata.obs.loc[adata.obs['primary_diagnosis'] == 'Other neurological disorder', 'primary_diagnosis'] = 'Healthy Control'
adata.obs['primary_diagnosis'] = adata.obs['primary_diagnosis'].cat.rename_categories({'Healthy Control': 'Control', 'Idiopathic PD': 'PD'})

# add info on different batches
# some donors have data from both PB and 10X and we have to take that into acccount for modelling
grouped = adata.obs.groupby(['vireo_assignment', 'batch']).size().reset_index(name='counts')
combined_dict = {}
for donor, group in grouped.groupby('vireo_assignment'):
    if group['counts'].gt(0).sum() > 1:
        combined_dict[donor] = 'mix'
    else:
        combined_dict[donor] = group.loc[group['counts'].gt(0), 'batch'].values[0]
adata.obs['batch_combined'] = adata.obs['vireo_assignment'].map(combined_dict)

# create pseudobulk
ps = pt.tl.PseudobulkSpace()
pdata = ps.compute(adata, target_col="vireo_assignment", groups_col="cell_type_simplified", layer_key="counts", mode="sum", min_cells=50, min_counts=1000)

# get cell type specific pseudobulk
cell_types = adata.obs['cell_type_simplified'].cat.categories

# print number of unique cell types
print(f"Number of unique cell types: {len(cell_types)}")
results = []
for cell_type in cell_types:
    n_donors = pdata[pdata.obs['cell_type_simplified'] == cell_type].obs['vireo_assignment'].nunique()
    results.append({
        "cell_type": cell_type,
        "n_unique_donors": n_donors
    })


summary_table = pd.DataFrame(results)
print(summary_table)



    


In [ ]:
summary_table.to_csv('cell_numbers/sn_number_of_donors.tsv', sep = '\t', index = False)

In [ ]:
# load adata
path_to_adata = '/02_combination_10x_pb/hdf5/20250811_all_data_integrated_filtered_umap_scANVI_metadata_ct_corrected.annot_corrected.index_dedup.cg.h5ad'
adata = sc.read_h5ad(path_to_adata)
adata.obs.set_index(adata.obs.new_index.astype(str),  inplace=True)

# prepare data for pseudobulk
adata.X = adata.layers["counts"]
adata = adata[~adata.obs['experiment'].isin(['mo005', 'mo009'])]
adata = adata[~adata.obs['region'].isin(['Motor Cortex'])]
adata.obs.loc[adata.obs['primary_diagnosis'] == 'Other neurological disorder', 'primary_diagnosis'] = 'Healthy Control'
adata.obs['primary_diagnosis'] = adata.obs['primary_diagnosis'].cat.rename_categories({'Healthy Control': 'Control', 'Idiopathic PD': 'PD'})

# add info on different batches
# some donors have data from both PB and 10X and we have to take that into acccount for modelling
grouped = adata.obs.groupby(['vireo_assignment', 'batch']).size().reset_index(name='counts')
combined_dict = {}
for donor, group in grouped.groupby('vireo_assignment'):
    if group['counts'].gt(0).sum() > 1:
        combined_dict[donor] = 'mix'
    else:
        combined_dict[donor] = group.loc[group['counts'].gt(0), 'batch'].values[0]
adata.obs['batch_combined'] = adata.obs['vireo_assignment'].map(combined_dict)

# create pseudobulk
ps = pt.tl.PseudobulkSpace()
pdata = ps.compute(adata, target_col="vireo_assignment", groups_col="consensus_cell_type_corrected", layer_key="counts", mode="sum", min_cells=50, min_counts=1000)

# get cell type specific pseudobulk
cell_types = adata.obs['consensus_cell_type_corrected'].cat.categories


results = []
for cell_type in cell_types:
    n_donors = pdata[pdata.obs['consensus_cell_type_corrected'] == cell_type].obs['vireo_assignment'].nunique()
    results.append({
        "cell_type": cell_type,
        "n_unique_donors": n_donors
    })


summary_table = pd.DataFrame(results)
print(summary_table)

In [ ]:
summary_table.to_csv('cell_numbers/cc_number_of_donors.tsv', sep = '\t', index = False)

In [ ]:
cell_types

In [ ]:
pip list

```
Package                      Version
---------------------------- --------------
absl-py                      2.1.0
adjustText                   1.3.0
aiohappyeyeballs             2.4.3
aiohttp                      3.10.10
aiosignal                    1.3.1
altair                       5.4.1
anndata                      0.10.8
ansicolors                   1.1.8
array_api_compat             1.9.1
arrow                        1.3.0
arviz                        0.20.0
asttokens                    2.4.1
astunparse                   1.6.3
async-timeout                4.0.3
attrs                        24.2.0
blitzgsea                    1.3.47
cached-property              1.5.2
certifi                      2024.8.30
cffi                         1.17.1
chardet                      5.2.0
charset-normalizer           3.4.0
chex                         0.1.87
click                        8.1.8
colorama                     0.4.6
comm                         0.2.2
contextlib2                  21.6.0
contourpy                    1.3.0
custom-inherit               2.4.1
cycler                       0.12.1
debugpy                      1.8.7
decorator                    5.1.1
decoupler                    1.8.0
docrep                       0.3.2
entrypoints                  0.4
equinox                      0.11.8
et_xmlfile                   2.0.0
ete3                         3.1.3
etils                        1.10.0
exceptiongroup               1.2.2
executing                    2.1.0
fastjsonschema               2.21.1
filelock                     3.16.1
flatbuffers                  24.3.25
flax                         0.10.1
fonttools                    4.54.1
formulaic                    1.0.2
frozenlist                   1.5.0
fsspec                       2024.10.0
gast                         0.6.0
get-annotations              0.1.2
gmpy2                        2.1.5
google-pasta                 0.2.0
grpcio                       1.67.1
h5netcdf                     1.4.0
h5py                         3.12.1
humanize                     4.11.0
idna                         3.10
igraph                       0.11.8
importlib_metadata           8.5.0
importlib_resources          6.4.5
interface-meta               1.3.0
ipykernel                    6.29.5
ipython                      8.29.0
jax                          0.4.35
jaxlib                       0.4.35
jaxopt                       0.8.3
jaxtyping                    0.2.34
jedi                         0.19.1
Jinja2                       3.1.4
joblib                       1.4.2
jsonschema                   4.23.0
jsonschema-specifications    2024.10.1
jupyter_client               8.6.3
jupyter_core                 5.7.2
keras                        3.6.0
kiwisolver                   1.4.7
lamin_utils                  0.13.7
legacy-api-wrap              1.4
legendkit                    0.3.4
leidenalg                    0.10.2
libclang                     18.1.1
lightning                    2.4.0
lightning-utilities          0.11.8
lineax                       0.0.7
llvmlite                     0.43.0
loguru                       0.7.2
Markdown                     3.7
markdown-it-py               3.0.0
MarkupSafe                   3.0.2
marsilea                     0.4.6
matplotlib                   3.9.2
matplotlib-inline            0.1.7
mdurl                        0.1.2
ml_collections               0.1.1
ml-dtypes                    0.4.1
mpmath                       1.3.0
msgpack                      1.1.0
mudata                       0.3.1
multidict                    6.1.0
multipledispatch             1.0.0
munkres                      1.1.4
muon                         0.1.7
namex                        0.0.8
narwhals                     1.13.2
natsort                      8.4.0
nbclient                     0.10.2
nbformat                     5.10.4
nest_asyncio                 1.6.0
networkx                     3.4.2
numba                        0.60.0
numpy                        1.26.4
numpyro                      0.15.3
nvidia-cublas-cu12           12.4.5.8
nvidia-cuda-cupti-cu12       12.4.127
nvidia-cuda-nvrtc-cu12       12.4.127
nvidia-cuda-runtime-cu12     12.4.127
nvidia-cudnn-cu12            9.1.0.70
nvidia-cufft-cu12            11.2.1.3
nvidia-curand-cu12           10.3.5.147
nvidia-cusolver-cu12         11.6.1.9
nvidia-cusparse-cu12         12.3.1.170
nvidia-nccl-cu12             2.21.5
nvidia-nvjitlink-cu12        12.4.127
nvidia-nvtx-cu12             12.4.127
openpyxl                     3.1.5
opt_einsum                   3.4.0
optax                        0.2.3
optree                       0.13.0
orbax-checkpoint             0.8.0
ott-jax                      0.4.9
packaging                    24.1
pandas                       2.2.3
papermill                    2.6.0
parso                        0.8.4
patsy                        0.5.6
pertpy                       0.9.4
pexpect                      4.9.0
pickleshare                  0.7.5
pillow                       11.0.0
pip                          24.3.1
platformdirs                 4.3.6
ply                          3.11
prompt_toolkit               3.0.48
propcache                    0.2.0
protobuf                     5.28.3
psutil                       6.1.0
ptyprocess                   0.7.0
PubChemPy                    1.0.4
pure_eval                    0.2.3
pyarrow                      18.0.0
pycairo                      1.27.0
pycparser                    2.22
pydeseq2                     0.4.12
Pygments                     2.18.0
PyGObject                    3.50.0
pynndescent                  0.5.13
Pyomo                        6.8.0
pyparsing                    3.2.0
pypng                        0.20220715.0
PyQt5                        5.15.11
PyQt5-Qt5                    5.15.15
PyQt5_sip                    12.15.0
pyro-api                     0.1.2
pyro-ppl                     1.9.1
python-dateutil              2.9.0
pytorch-lightning            2.4.0
pytz                         2024.1
PyYAML                       6.0.2
pyzmq                        26.2.0
referencing                  0.35.1
reportlab                    4.2.5
requests                     2.32.3
rich                         13.9.4
rpds-py                      0.20.1
rpy2                         3.5.16
scanpy                       1.10.3
schist                       0.9.1
scikit-learn                 1.5.2
scikit-misc                  0.5.1
scipy                        1.14.1
scvi-tools                   1.2.0
seaborn                      0.13.2
session-info                 1.0.0
setuptools                   75.3.0
six                          1.16.0
sparse                       0.15.4
sparsecca                    0.3.1
stack-data                   0.6.2
statsmodels                  0.14.4
stdlib-list                  0.11.0
sympy                        1.13.1
tenacity                     9.0.0
tensorboard                  2.18.0
tensorboard-data-server      0.7.2
tensorflow                   2.18.0
tensorflow-io-gcs-filesystem 0.37.1
tensorstore                  0.1.67
termcolor                    2.5.0
texttable                    1.7.0
threadpoolctl                3.5.0
toolz                        1.0.0
torch                        2.5.1
torchmetrics                 1.5.1
tornado                      6.4.1
toyplot                      2.0.0
toytree                      3.0.6
tqdm                         4.66.6
traitlets                    5.14.3
triton                       3.1.0
typeguard                    2.13.3
types-python-dateutil        2.9.0.20241003
typing_extensions            4.12.2
tzdata                       2024.2
tzlocal                      5.2
umap-learn                   0.5.7
unicodedata2                 15.1.0
urllib3                      2.2.3
wcwidth                      0.2.13
Werkzeug                     3.1.2
wheel                        0.44.0
wrapt                        1.16.0
xarray                       2024.10.0
xarray-einstats              0.8.0
yarl                         1.17.1
zipp                         3.20.2
zstandard                    0.23.0
Note: you may need to restart the kernel to use updated packages.
```
